# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHAAkash/-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring**, with a classification model underneath as the mechanism that
produces the score.

The decision this serves is "which pages should a reviewer look at first" — that is a ranking
question ("which ones first?"), not a plain yes/no question. But the cleanest way to *produce* a
ranking is to first estimate a probability for something concrete and checkable (does this page
look like it's declining while still carrying demand?), then sort pages by that probability. So
under the hood this is binary classification (`is_declining_label` vs not), and the ranking is
the classifier's predicted probability turned into an ordered list. This mirrors the starter
pipeline's own design (`scripts/03_train_model.py` trains a classifier; `04_evaluate_and_export.py`
turns its probabilities into a ranked, reason-coded queue) — I'm not copying its label choice
blindly, just its shape: classify to score, score to rank.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Why "ranking/scoring" and not plain classification: the trend_direction column already shows
# this isn't a clean binary world -- there are 5 categories, and a reviewer needs an ORDER
# across all of them (a page 90% likely to decline vs 55% likely still both matter, just less
# urgently), not a single cutoff.
print(df["trend_direction"].value_counts())


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (for now): `is_declining_label = (trend_direction == "down")`** — the same proxy the
starter pipeline uses. I'm naming it honestly as a **proxy, not an observed future outcome**:
`trend_direction` is itself computed from `trend_pct`, which is a bucket calculated from the
*current* 90-day window, not something that happens *after* a decision point. That means
`trend_direction` and `trend_pct` can never be features (they'd leak the label into itself), and
it also means this label describes "what the page's recent trend bucket already says," not "what
will happen next" — a real weakness the lane guide calls out directly (section 5).

**Why I'm using it anyway, for this notebook:** it's the fastest way to get a working, checkable
task framed this week, and it's exactly what the starter pipeline's own validated numbers
(`outputs/model_report.md`) are measured against, so I have an honest baseline to compare
against once I build my own version.

**What I'd change before the real capstone claim:** move to a genuinely future-looking target —
`features from the prior 90 days -> decline over the next 30 days`, built from the warehouse's
daily fact table (`fact_content_daily_performance`) with a strict prior-window/future-window
split, so the label is an observed outcome instead of a same-window bucket. I'll revisit this in
the leakage-audit assignment before locking in a final label.


In [2]:
# The label trap, made concrete: is_declining_label comes from trend_direction, which comes
# from trend_pct -- so both trend_direction and trend_pct are OFF LIMITS as features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"declining-label rate: {df['is_declining_label'].mean():.3f}")
print(f"declining-label count: {df['is_declining_label'].sum():,} of {len(df):,}")
print()
print("Columns that must be EXCLUDED from features because they define the label:")
print(" - trend_direction  (the label's direct source)")
print(" - trend_pct        (trend_direction is computed from this)")


declining-label rate: 0.542
declining-label count: 16,262 of 30,000

Columns that must be EXCLUDED from features because they define the label:
 - trend_direction  (the label's direct source)
 - trend_pct        (trend_direction is computed from this)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: precision@50** — of the top 50 pages the ranking puts first, how many actually
carry the label. I'm defending this over plain accuracy or ROC-AUC because of how the output gets
*used*: a reviewer works down a ranked queue and only ever reaches the top of it, not the whole
30,000-row inventory. A model can have great overall accuracy while its top 50 are mediocre if
the "easy" correct predictions are all buried in the boring middle of the ranking — precision@50
is the number that actually reflects what a reviewer experiences in week one.

**What "good" looks like, concretely:** the base rate of `is_declining_label` is 54.2% — so a
random ranking would already get roughly 27 of its top 50 "right" by chance. A ranking is only
earning its keep if it clears that random floor by a wide, defensible margin. The starter
pipeline's own random-forest model reaches 0.740 precision@50 (37 of 50) against a 0.240 baseline
rule (12 of 50) — I'll treat something in that neighborhood, measured with proper client-holdout
validation, as my working definition of "good" until I have reason to move the bar.


In [3]:
# Context for judging precision@50: what would "no skill" look like on this label?
base_rate = df["is_declining_label"].mean()
random_top50_hits = round(base_rate * 50)
print(f"label base rate: {base_rate:.3f}")
print(f"expected hits in top 50 from a RANDOM ranking: ~{random_top50_hits} of 50")
print()
print("starter pipeline reference (client-holdout, from outputs/model_report.md):")
print("  baseline rules     precision@50 = 0.240  (~12 of 50)")
print("  random forest      precision@50 = 0.740  (~37 of 50)")


label base rate: 0.542
expected hits in top 50 from a RANDOM ranking: ~27 of 50

starter pipeline reference (client-holdout, from outputs/model_report.md):
  baseline rules     precision@50 = 0.240  (~12 of 50)
  random forest      precision@50 = 0.740  (~37 of 50)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page)**, identified by `content_id`, nested under one `client_id`.
This is the grain the reviewer actually acts on — nobody refreshes "a client," they refresh one
specific page. Below is the actual slice I'd rank: eligible pages (enough impressions to matter,
old enough to have a real trend), with the columns that would feed the model and the columns that
explain *why* a page scored the way it did.


In [4]:
# Eligibility filter, matching the starter pipeline's own rule: enough impressions to be
# more than noise, and old enough to have a real trend rather than a "new" content burst.
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

view_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position",
    "days_since_last_update", "word_count", "content_age_days", "is_declining_label",
]
print(f"eligible rows: {len(eligible):,} of {len(df):,} total")
eligible[view_cols].head(5)


eligible rows: 30,000 of 30,000 total


,content_id,client_id,impressions_90d,clicks_90d,avg_position,days_since_last_update,word_count,content_age_days,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,20,3221.0,187,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,25,2481.0,445,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,20,3515.0,141,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,22,NaN,463,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,14,2803.0,263,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

No single signal predicts the label on its own — see the correlations below. `word_count`
(+0.09), `days_since_last_update` (+0.08), and `avg_position` (-0.03) each nudge the label a
little, but none of them, alone, would make a defensible if-statement ("flag if word_count is
low" would misfire constantly). A fixed rule has to pick one or two thresholds and hope; a model
can weigh many weak, tangled signals — position, freshness, depth, historical impression days,
content type — together, and the starter pipeline's own numbers show that actually working: the
hand-written baseline rule gets 0.240 precision@50, while a random forest using the same signals
gets 0.740. That gap is exactly what "the pattern is real but too messy to write by hand" looks
like in practice, not just in theory.

That said, a rule isn't worthless here — it's still the right *first* thing to build, because it's
transparent and gives me something honest to beat. ML earns its place only after the rule is
built and shown to leave real gains on the table, which the starter numbers already show it does.


In [5]:
# Individual signal strength -- none of these alone would make a trustworthy rule.
corr_cols = ["impressions_90d", "avg_position", "days_since_last_update", "word_count",
             "engagement_rate", "scroll_rate"]
correlations = df[corr_cols + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"]
correlations = correlations.drop("is_declining_label").sort_values()
print("correlation of each single signal with the label (none is strong alone):")
print(correlations)


correlation of each single signal with the label (none is strong alone):
avg_position             -0.029035
impressions_90d          -0.018175
engagement_rate          -0.012743
scroll_rate              -0.002958
days_since_last_update    0.081383
word_count                0.090157
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.